# Notebook 08 (optional) — Analyst-recommendations basket

*Portfolio Intelligence Engine — User Guide Series (optional standalone chapter).*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
issue [#1367](https://github.com/prajoria/OpenBB/issues/1367).

---

## Where this notebook fits

The main 7-notebook series (NB01→NB07) teaches how to *analyze* a basket.
It never teaches how to *construct one from scratch*. Readers who finish
NB07 always ask the same two questions:

1. The 10-position through-line basket is synthetic — what would a
   real, defensible basket look like if I sat down with a blank page?
2. Which sources do actual traders use? Where do I go for signal
   beyond `fmp_cached`?

This standalone notebook answers both, in one file. It reads
independently of the through-line basket — NB08 does not require any
`.notebook_state/` artifact from NB01-NB07 to run.

By the end we can answer:

> *If I built a 15-ETF all-weather basket using published discipline
> (Dalio, Bogle, Faber, Fidelity), what would the review numbers look
> like, and what would the reader do next?*

**Not a stock-picking recommendation.** Everything in this notebook is
educational — no forward-return targets, no "buy this." Illustrative
weights only.


### Provider chain for this notebook (Track A / [#1436](https://github.com/prajoria/OpenBB/issues/1436))

The same 5-tier chain from [NB01 §2](./01-getting-started-and-providers.ipynb):

> **fmp_cached → fmp → cboe → sec (EDGAR) → yfinance (last-resort, personal-use)**

For NB08's specific data paths:

| Data path used below | Primary | Free-authoritative fallback |
|---|---|---|
| ETF constituent look-through (all 16 ETFs) | **SEC N-PORT** (`sec`) | *authoritative-primary since PR [#1445](https://github.com/prajoria/OpenBB/issues/1445) (#1426)* |
| Prices / OHLCV for backtest | `fmp_cached` | `cboe` (EOD) |
| Sector classification per constituent | `fmp_cached` (`obb.equity.profile`) | `sec` industry code → *"Unknown"* bucket |
| Analyst grades / price targets | `fmp_cached` | *no free authoritative source — gap; keep on `fmp`* |
| Options / IV context (if referenced) | `cboe` `OptionsChains` | n/a — cboe IS the exchange |

**Non-N-PORT filers**: GLD (commodity grantor trust) and DBC (commodity
pool) pass through the `NportUnavailable` branch as opaque single-symbol
positions — the concentration math treats them like any other line item;
sector classification just shows "Commodities / Other".

Zero code cells change under this PR — the SEC N-PORT migration for §5
+ the low-cost 5-ETF Variant B (§10) both shipped in prior PRs
(#1445 (#1426), [#1424](https://github.com/prajoria/OpenBB/issues/1424)).


In [ ]:
# [NB08 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python version           {sys.version.split()[0]}")
print(f"venv sanity check:       passed")
print(f"State dir (repo-rel):    {STATE}/")


Python version           3.12.10
venv sanity check:       passed
State dir (repo-rel):    .notebook_state/


In [ ]:
# [NB08] shared HTML-rendering toolkit (same as NB02–NB07)
# Import the notebook-render helpers so §-cells below emit styled panels
# instead of plain print(). Compute logic is unchanged — only the
# presentation of results is upgraded to the shared toolkit.
import sys, pathlib as _pl
_here = _pl.Path.cwd()
for _p in (_here, _here / "notebooks" / "portfolio"):
    if (_p / "_nb_render.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
from _nb_render import (
    nb_panel, nb_table, nb_pill, nb_render_phase, nb_toolkit_legend, NB_LINKS,
)
nb_toolkit_legend()

## 1. Why this notebook + reader contract

This is the *construction* notebook the main series doesn't have.
NB03 taught me how to x-ray a basket. NB05 taught me to attribute
returns. NB06 taught me to backtest. What none of them taught me: how
to sit down with 15 empty slots and fill them with a defensible
process rather than my last three CNBC-driven hunches.

The trader's problem: when I first tried to build a "proper"
portfolio, I couldn't tell whether my weights came from a framework
or from vibes. The frameworks *exist* — Dalio, Bogle, Faber,
Fidelity all published their playbooks — but nobody hands the working
retail trader a synthesized "here's how they compose." So I did the
synthesis myself in §2, cited every framework line-by-line, and then
ran the resulting basket through the same review NB03-NB06 would run
on any other book.

**What you get out of this notebook:**

- A 15-ETF basket with each position's weight tied to a specific
  published framework (Dalio, Bogle, Faber, Fidelity).
- A look-through view that decomposes it into ~2,500 underlying
  positions across 11 GICS sectors.
- HHI + effective-N so you can quote concentration in one sentence.
- A backtest over 2023-01-03 → 2024-12-31 vs SPY.
- One code cell showing where to plug in your own "Fortress"
  reference basket if you already have one.
- A curated set of analyst-research destinations (TipRanks, Zacks,
  Morningstar, Seeking Alpha, ETF.com, SEC EDGAR) with what each
  actually gives you.

**What you do NOT get:** live scraping of analyst sites, a
Bridgewater 13F pull, live broker execution, or a claim that this
basket will make money. That is beyond scope by design (see §11).


## 2. Portfolio construction — the 5 frameworks

Every seat at the retail table borrows from one of these. Even
active managers who claim to be "bottom-up stock pickers" have an
implicit framework — usually a modified version of Fidelity's
sector-rotation lens filtered through their own conviction. Naming
them explicitly lets you tell your framework apart from your
hunches.

> **📖 Risk parity** — allocate capital so each asset contributes
> equal *risk* (not equal dollars) to the portfolio. Bonds get a
> larger dollar weight because they're less volatile; equities get a
> smaller dollar weight because they carry more of the variance.
> Dalio's All Weather is the best-known example. [Investopedia →](https://www.investopedia.com/terms/r/risk-parity.asp)
>
> **📖 All-weather / all-seasons portfolio** — Ray Dalio's four-quadrant
> macro framework (growth up/down × inflation up/down) with a
> 30/40/15/7.5/7.5 target allocation across stocks, long bonds,
> intermediate bonds, gold, and commodities. Designed to survive any
> macro regime without predicting which one is next. [Investopedia →](https://www.investopedia.com/terms/a/all-weather-fund.asp)
>
> **📖 Three-fund portfolio** — the Bogleheads' minimalist template:
> total US stock market + total international + total bond. Rebalance
> annually, keep expense ratios under 10 bps. The reference "have I
> beaten this?" bar for any active retail strategy. [Investopedia →](https://www.investopedia.com/terms/t/three-fund-portfolio.asp)
>
> **📖 Sector rotation** — Fidelity's business-cycle framework: at
> each stage of the cycle (early / mid / late / recession), specific
> GICS sectors historically lead. Early = consumer discretionary,
> financials; mid = technology, industrials; late = energy,
> materials; recession = consumer staples, utilities, health care.
> Rotate exposure with the cycle. [Investopedia →](https://www.investopedia.com/terms/s/sector-rotation.asp)
>
> **📖 Target-date fund** — Vanguard's glide-path approach: an
> age-anchored fund whose equity share glides down and bond share
> glides up as you approach the target retirement year. Used here
> only as the "starting point" reference for a mid-career investor
> (60/40-ish equity/bond mix). [Investopedia →](https://www.investopedia.com/terms/t/target-date_fund.asp)
>
> **📖 Trend following** — a rules-based overlay that goes long an
> asset only when it's above its 10-month moving average (Faber's
> Ivy variant). Not a *construction* framework — it's an *exit*
> framework layered on top. Cited here for completeness; NB08 does
> not implement the overlay. [Investopedia →](https://www.investopedia.com/articles/active-trading/091714/basics-trend-following.asp)

The 15-position basket in §4 is a **synthesis** — no one framework
above dominates. Broad market spine + fixed-income ballast is
Bogleheads; the gold/long-bond/commodity kickers are Dalio; the
sector-satellite tilts are Fidelity; and the equal-weight approach
across asset classes is Faber. Every row in §4 cites the framework
it came from so you can substitute your own choices with your own
citations.


## 3. Trader / analyst sources — where the pros look

`fmp_cached` gets us prices, fundamentals, key metrics, ratios, and
financial scores. It does not get us **published analyst opinion** —
price targets, ratings, moat scores, sentiment aggregations. Those
live on dedicated aggregator sites. Cited here so you know where to
click; NB08 does not scrape them (that's future work, and each site
has a distinct ToS).

> **📖 Analyst price target** — the 12-month forward price forecast
> a covering analyst publishes with a rating (Buy / Hold / Sell). A
> consensus target is the mean or median across all covering
> analysts. Directional signal only — the *distribution* of targets
> tells you more than the mean, because a tight cluster and a wide
> cluster have very different information content. [Investopedia →](https://www.investopedia.com/terms/p/pricetarget.asp)
>
> **📖 Zacks Rank** — a 1-to-5 ranking (1 = Strong Buy, 5 = Strong
> Sell) computed daily from earnings-estimate revisions. Zacks
> claims a Rank-1 basket has outperformed the S&P over their
> lookback; treat that claim skeptically (survivorship + universe
> definition matter) but the underlying signal — estimate revisions
> — is a well-documented factor. [Investopedia →](https://www.investopedia.com/terms/z/zacks-lifecycle.asp)
>
> **📖 Morningstar star rating** — a 1-5 star rating based on a
> fund's past risk-adjusted return vs its category peers. Backward-
> looking; useful for screening OUT bottom-quintile funds, less
> useful for picking future top performers (per Morningstar's own
> research). Their forward-looking Medalist rating (Gold/Silver/
> Bronze) is a separate, analyst-driven overlay. [Investopedia →](https://www.investopedia.com/terms/m/morningstar-risk-rating.asp)
>
> **📖 SEC EDGAR** — the free, official filings database for every
> US-listed security. 13F holdings, 10-K annual reports, S-1
> registrations, insider Form 4s — all here. Bridgewater,
> Renaissance, Berkshire, ARK all file 13Fs; you can read the
> actual holdings 45 days after each quarter-end. [Investopedia →](https://www.investopedia.com/terms/e/edgar.asp)
>
> **📖 Commodity ETF** — an exchange-traded fund that holds either
> physical commodities (GLD, SLV) or commodity futures (DBC, USO).
> Used in this basket as an inflation and macro-regime hedge per
> the Dalio framework. Tax treatment varies — check the K-1 vs
> 1099 status before adding to a taxable account. [Investopedia →](https://www.investopedia.com/terms/c/commodity-etf.asp)

**Six destinations worth bookmarking** — what each actually gives you
that the others don't:

| Site | What it aggregates | URL |
|---|---|---|
| TipRanks | Aggregated analyst price targets + insider/hedge-fund sentiment scores | https://www.tipranks.com/ |
| Zacks Investment Research | Zacks Rank (1-5) based on earnings-estimate revisions | https://www.zacks.com/ |
| Morningstar | Star ratings + fair-value estimates + moat analysis | https://www.morningstar.com/ |
| Seeking Alpha | Crowd-sourced analyst articles + Quant Ratings | https://seekingalpha.com/ |
| ETF.com | ETF-specific analyst reports, expense-ratio comparisons | https://www.etf.com/ |
| SEC EDGAR — 13F filings | Quarterly institutional holdings (Bridgewater, Renaissance, Berkshire, ARK…) | https://www.sec.gov/edgar/searchedgar/companysearch |


## 4. The 15-ETF basket — table + rationale

**Variant A — Diversified all-weather (16 ETFs, higher maintenance).**


Synthesis of §2's frameworks. Every row cites which framework
contributes it. Weights sum to 100%.

**Broad market spine (55%)** — Bogleheads three-fund plus Faber's
REIT + Dalio's gold sleeve.

**Fixed-income ballast (25%)** — Bogleheads total-bond plus Dalio's
long / short / TIPS split.

**Sector satellites (20%)** — Fidelity sector-rotation tilts that
overweight the areas VTI is structurally light in
(defensives + inflation-sensitive + EM diversifier).

> **📖 Rebalancing** — the discipline of periodically trimming
> positions that have grown above their target weight and adding to
> those that have shrunk below. Enforces "buy low, sell high" on your
> own book automatically. For a basket like this: annual rebalance if
> in a taxable account, monthly if tax-sheltered. [Investopedia →](https://www.investopedia.com/terms/r/rebalancing.asp)
>
> **📖 Efficient frontier** — the Markowitz curve of all portfolios
> that maximize expected return for a given level of variance. Every
> framework in §2 is an opinionated point on (or near) this frontier;
> none of them *is* the frontier itself, because the frontier requires
> a forecast of expected returns you don't actually have. [Investopedia →](https://www.investopedia.com/terms/e/efficientfrontier.asp)

*The code cell below writes the basket to
`.notebook_state/analyst_basket.json` in the same shape as
`basket.json` and prints the table.*


In [ ]:
# [NB08 §4] Build the 15-ETF basket + write JSON
import json
from pathlib import Path
from IPython.display import display, HTML

BASKET = [
    # Broad market spine (55%)
    {"symbol": "VTI",  "weight": 0.30, "role": "US equity broad",        "framework": "Bogleheads three-fund"},
    {"symbol": "VXUS", "weight": 0.15, "role": "Ex-US developed + EM",   "framework": "Bogleheads three-fund"},
    {"symbol": "VNQ",  "weight": 0.05, "role": "REITs",                  "framework": "Faber Ivy"},
    {"symbol": "GLD",  "weight": 0.05, "role": "Inflation hedge (gold)", "framework": "Dalio all-weather"},
    # Fixed-income ballast (25%)
    {"symbol": "BND",  "weight": 0.15, "role": "Investment-grade agg",   "framework": "Bogleheads three-fund"},
    {"symbol": "TLT",  "weight": 0.05, "role": "Long-duration hedge",    "framework": "Dalio all-weather"},
    {"symbol": "SHY",  "weight": 0.03, "role": "Short-duration liquidity","framework": "Dalio all-weather"},
    {"symbol": "TIP",  "weight": 0.02, "role": "Real-rate hedge (TIPS)", "framework": "Dalio all-weather"},
    # Sector satellites (20%)
    {"symbol": "XLE",  "weight": 0.03, "role": "Energy sector",          "framework": "Fidelity rotation"},
    {"symbol": "XLF",  "weight": 0.03, "role": "Financials sector",      "framework": "Fidelity rotation"},
    {"symbol": "XLV",  "weight": 0.03, "role": "Health care sector",     "framework": "Fidelity rotation (defensive)"},
    {"symbol": "XLU",  "weight": 0.02, "role": "Utilities sector",       "framework": "Fidelity rotation (defensive)"},
    {"symbol": "XLB",  "weight": 0.02, "role": "Materials sector",       "framework": "Fidelity rotation"},
    {"symbol": "XLI",  "weight": 0.02, "role": "Industrials sector",     "framework": "Fidelity rotation"},
    {"symbol": "DBC",  "weight": 0.03, "role": "Broad commodities",      "framework": "Dalio all-weather + Faber Ivy"},
    {"symbol": "VWO",  "weight": 0.02, "role": "Emerging markets equity","framework": "Faber Ivy diversifier"},
]

total_w = sum(p["weight"] for p in BASKET)
assert abs(total_w - 1.0) < 1e-9, f"weights sum to {total_w}, not 1.0"

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
basket_path = state / "analyst_basket.json"
basket_path.write_text(json.dumps(BASKET, indent=2), encoding="utf-8")

# Convenience alias used in later cells
basket = BASKET

# ---- render ----
_rows = [
    [nb_pill(p["symbol"], "accent"), f"{p['weight']*100:.1f}%", p["role"], p["framework"]]
    for p in BASKET
]
display(HTML(nb_panel(
    "The 15-ETF diversified basket",
    nb_table(["ticker", "weight", "role", "framework"], _rows),
    subtitle=f"Wrote .notebook_state/analyst_basket.json ({len(BASKET)} ETFs, weights sum to "
             f"{total_w*100:.1f}%). A textbook multi-asset allocation: a broad-market spine, "
             "fixed-income ballast, and sector satellites — drawn from the Bogleheads three-fund, "
             "Dalio all-weather, and Fidelity sector-rotation frameworks.",
    tone="good", badge=f"{len(BASKET)} ETFs",
    links=[NB_LINKS["etf"], NB_LINKS["asset_allocation"], NB_LINKS["three_fund"],
           NB_LINKS["all_weather"], NB_LINKS["diversification"]],
)))

ticker,weight,role,framework
VTI,30.0%,US equity broad,Bogleheads three-fund
VXUS,15.0%,Ex-US developed + EM,Bogleheads three-fund
VNQ,5.0%,REITs,Faber Ivy
GLD,5.0%,Inflation hedge (gold),Dalio all-weather
BND,15.0%,Investment-grade agg,Bogleheads three-fund
TLT,5.0%,Long-duration hedge,Dalio all-weather
SHY,3.0%,Short-duration liquidity,Dalio all-weather
TIP,2.0%,Real-rate hedge (TIPS),Dalio all-weather
XLE,3.0%,Energy sector,Fidelity rotation
XLF,3.0%,Financials sector,Fidelity rotation


<IPython.core.display.HTML object>

## 5. Basket X-Ray — look-through via SEC N-PORT

Same discipline as NB03, migrated to authoritative filings. Each equity ETF
gets unwrapped via its **SEC Form N-PORT** disclosure
(`obb.etf.nport_disclosure(symbol=..., provider="sec")`) — the quarterly
holdings filing every '40-Act US-registered fund is required to make with
the SEC. Each bond ETF gets the same treatment: N-PORT covers bond funds
too, so BND / TLT / TIP / SCHP unwrap to the same free, redistribution-safe
government-filing feed instead of a Yahoo-scraped bond ladder. Commodity
grantor trusts (GLD, DBC) pass through as-is — they file 10-K/8-K, not
N-PORT, so the look-through helper flags them as `NportUnavailable` and the
notebook leaves them opaque without erroring.

The switch was tracked as [#1426](https://github.com/prajoria/OpenBB/issues/1426) (notebook consumer) — the notebook-facing
half of the data-provenance concern in [#1425](https://github.com/prajoria/OpenBB/issues/1425) (Yahoo-shaped snapshots in a
public repo). N-PORT is free, US-government, redistribution-safe, and
authoritative-because-filed-by-the-issuer. The tradeoff is **freshness**:
N-PORT is quarterly-visible with a ~30-60 day lag vs Yahoo's live page, so
the constituent list you see reflects the fund's position on its most recent
quarter-end, not last Friday. For a portfolio-construction notebook (this
one) that lag is a rounding error — VTI's top-50 does not turn over in a
quarter. For an intraday tool it would matter; use the fmp_cached
`obb.etf.holdings` route there instead.

The point of look-through on this basket: to check whether the
sector-satellite tilts in §4 actually move the effective sector weights, or
whether VTI's 30% concentration in the top-10 US names dwarfs the
satellites and swallows the whole tilt.

Every N-PORT fetch disk-caches under `.notebook_state/nport_cache/` so a
kernel-restart re-run does not re-hit SEC EDGAR.

In [ ]:
# [NB08 §5] Look-through via SEC N-PORT (replaces yfinance snapshot path — GH #1426)
# Cache is per-operator under .notebook_state/nport_cache/ (gitignored),
# populated on first run.
import sys
sys.path.insert(0, ".")
from _nport_lookthrough import effective_positions, NportUnavailable
from IPython.display import display, HTML

EQUITY_ETFS = {
    "VTI", "VXUS", "VNQ", "VWO",
    "XLE", "XLF", "XLV", "XLU", "XLB", "XLI",
    "QQQ", "SPY", "DIA", "IWM", "VOO", "VEA",
}
BOND_ETFS = {"BND", "TLT", "SHY", "TIP", "SCHP", "AGG"}
COMMODITY_TRUSTS = {"GLD", "DBC", "SLV"}

effective, non_nport, opaque = effective_positions(
    basket,
    equity_etfs=EQUITY_ETFS,
    bond_etfs=BOND_ETFS,
    commodity_trusts=COMMODITY_TRUSTS,
    top_n_per_etf=50,   # top 50 issuers per ETF; tail bucketed as TAIL_<ETF>
)

# ---- render ----
_top_rows = [
    [key[:38], f"{w*100:.2f}%"]
    for key, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]
]
_body = nb_table(["issuer / bucket", "weight"], _top_rows)
if non_nport:
    _opaque_rows = [[nb_pill(sym, "warn"), "commodity grantor trust or non-'40-Act fund"]
                    for sym in non_nport]
    _body += (
        "<div style='margin:9px 0 6px;font:600 11px ui-sans-serif,system-ui;opacity:.7;"
        "text-transform:uppercase;letter-spacing:.03em'>Non-N-PORT filers (left opaque)</div>"
        + nb_table(["symbol", "reason"], _opaque_rows)
    )
display(HTML(nb_panel(
    "Basket X-ray — SEC N-PORT look-through",
    _body,
    subtitle=f"{len(basket)} ETFs flatten to {len(effective)} distinct effective positions "
             f"(total effective weight {sum(effective.values())*100:.1f}%). N-PORT is the SEC's "
             "quarterly fund-holdings disclosure — flattening the ETFs into their constituents "
             "reveals the true issuer-level concentration a naive ticker count hides.",
    tone="accent", badge=f"{len(effective)} positions",
    links=[NB_LINKS["look_through"], NB_LINKS["nport"], NB_LINKS["etf"], NB_LINKS["diversification"]],
)))

issuer / bucket,weight
TAIL_VTI,13.98%
TAIL_BND,12.55%
TAIL_VXUS,10.91%
GLD,5.00%
BOND_TLT,4.96%
DBC,3.00%
BOND_BND,2.40%
BOND_TIP,2.00%
BOND_SHY,1.99%
NVIDIA Corp,1.93%


<IPython.core.display.HTML object>

## 6. Risk metrics — HHI + effective-N + sector view

Two concentration numbers to quote from now on:

- **HHI** (Herfindahl-Hirschman Index — see NB03 §5) — sum of
  squared weights, 1/N to 1.
- **Effective-N** — 1 / HHI. "How many equivalent equal-weight
  positions am I really holding?"

Naive vs look-through sector view answers the question §4 dodged:
did the sector satellites actually move sector weights, or did the
55% broad-market spine swallow the tilts?

*The code cell below computes all four numbers.*


In [ ]:
# [NB08 §6] HHI + effective-N + sector view (naive vs look-through)
# Sector rollup on the flattened positions uses SEC issuer-name → ticker
# resolution (obb.equity.search, provider="sec"), then ticker → sector via
# fmp_cached equity.profile. Both cached under .notebook_state/nport_cache/
# so a kernel-restart re-run is cheap.
import sys
sys.path.insert(0, ".")
from _nport_lookthrough import nport_holdings, sector_rollup, NportUnavailable
from openbb import obb
import warnings; warnings.filterwarnings("ignore")
from IPython.display import display, HTML

def hhi(weights):
    return sum(w*w for w in weights)

# Naive HHI + N over the 15 ETFs
raw_weights = [p["weight"] for p in basket]
hhi_raw = hhi(raw_weights)
neff_raw = 1.0 / hhi_raw

# Look-through HHI + N over the effective flattened positions
xray_weights = list(effective.values())
hhi_xray = hhi(xray_weights)
neff_xray = 1.0 / hhi_xray

# Sector view — naive (each ETF is one bucket)
NAIVE_SECTOR = {
    "VTI": "Broad US Equity", "VXUS": "Broad Intl Equity",
    "VNQ": "Real Estate ETF", "GLD": "Commodity (Gold)",
    "BND": "Bond Fund", "TLT": "Bond Fund", "SHY": "Bond Fund",
    "TIP": "Bond Fund", "SCHP": "Bond Fund",
    "XLE": "Energy", "XLF": "Financials", "XLV": "Health Care",
    "XLU": "Utilities", "XLB": "Materials", "XLI": "Industrials",
    "DBC": "Commodity (Broad)", "VWO": "Broad EM Equity",
}
naive_by_sector = {}
for p in basket:
    sec = NAIVE_SECTOR.get(p["symbol"], "Unknown")
    naive_by_sector[sec] = naive_by_sector.get(sec, 0.0) + p["weight"]

# Sector view — look-through via N-PORT + SEC name-search rollup
EQUITY_ETFS = {"VTI","VXUS","VNQ","VWO","XLE","XLF","XLV","XLU","XLB","XLI","QQQ"}
BOND_ETFS = {"BND","TLT","SHY","TIP","SCHP"}
COMMODITY_TRUSTS = {"GLD","DBC"}

xray_by_sector: dict[str, float] = {}
unknown_count = 0
for pos in basket:
    sym = pos["symbol"]; w = pos["weight"]
    if sym in COMMODITY_TRUSTS:
        bucket = "Commodity"
        xray_by_sector[bucket] = xray_by_sector.get(bucket, 0.0) + w
        continue
    if sym in BOND_ETFS:
        bucket = "Fixed Income"
        xray_by_sector[bucket] = xray_by_sector.get(bucket, 0.0) + w
        continue
    if sym in EQUITY_ETFS:
        try:
            rows = nport_holdings(sym)
        except NportUnavailable:
            xray_by_sector[sym] = xray_by_sector.get(sym, 0.0) + w
            continue
        # rollup: scale weights by this ETF's weight in the basket
        sec_w, cnt = sector_rollup(rows, weight_scale=w, top_n_resolve=30)
        for k, v in sec_w.items():
            xray_by_sector[k] = xray_by_sector.get(k, 0.0) + v
        unknown_count += cnt.get("Unknown", 0) + cnt.get("Other (small)", 0)
    else:
        # single-name equity — use its own sector
        try:
            info = obb.equity.profile(symbol=sym, provider="fmp_cached").results
            sec = getattr(info[0], "sector", None) if info else "Unknown"
        except Exception:
            sec = "Unknown"
        sec = sec or "Unknown"
        xray_by_sector[sec] = xray_by_sector.get(sec, 0.0) + w

# ---- render ----
_conc_rows = [
    ["HHI", f"{hhi_raw:.4f}", f"{hhi_xray:.4f}",
     nb_pill(f"{hhi_xray-hhi_raw:+.4f}", "bad" if hhi_xray > hhi_raw else "good")],
    ["Effective-N", f"{neff_raw:.2f}", f"{neff_xray:.2f}",
     nb_pill(f"{neff_xray-neff_raw:+.2f}", "good" if neff_xray > neff_raw else "bad")],
]
_body = nb_table(["concentration", "naive", "x-ray", "Δ"], _conc_rows)
_naive_rows = [[sec, f"{w*100:.1f}%"]
               for sec, w in sorted(naive_by_sector.items(), key=lambda kv: -kv[1])[:6]]
_body += (
    "<div style='margin:9px 0 6px;font:600 11px ui-sans-serif,system-ui;opacity:.7;"
    "text-transform:uppercase;letter-spacing:.03em'>Naive sector view (top 6)</div>"
    + nb_table(["sector", "weight"], _naive_rows)
)
_xray_rows = [[sec, f"{w*100:.2f}%"]
              for sec, w in sorted(xray_by_sector.items(), key=lambda kv: -kv[1])[:8]]
_body += (
    "<div style='margin:9px 0 6px;font:600 11px ui-sans-serif,system-ui;opacity:.7;"
    "text-transform:uppercase;letter-spacing:.03em'>Look-through sector view (top 8, via N-PORT + SEC name search)</div>"
    + nb_table(["sector", "weight"], _xray_rows)
)
display(HTML(nb_panel(
    "Concentration — naive count vs. look-through reality",
    _body,
    subtitle=f"Rollup transparency: {unknown_count} constituents bucketed as Unknown / Other "
             "(small) across the equity ETFs (tail rows below the top-30 resolution threshold; "
             "contribute <0.05% each). HHI and Effective-N reveal whether flattening the ETFs "
             "concentrates or diversifies the true issuer-level exposure.",
    tone="neutral", badge="naive vs x-ray",
    links=[NB_LINKS["hhi"], NB_LINKS["effective_n"], NB_LINKS["look_through"], NB_LINKS["diversification"]],
)))

<IPython.core.display.HTML object>

## 7. Single-name spot-check — Analysis 7-phase on the largest holding

VTI's top constituent for years has been MSFT. Running the full
Analysis pipeline on MSFT gives us a per-name conviction score for
the single position that dominates the look-through view. This is
exactly the discipline NB02 demonstrated — the notebook cheats by
picking a name we already know the pipeline handles cleanly
(fixture-locked) so this cell is deterministic under kernel restart.

> **📖 Fundamental analysis** — valuing a security from its
> underlying financial statements, industry position, and management
> quality, as opposed to its price chart. The 7-phase Analysis
> pipeline is a fundamentals-first workflow; the technicals only
> enter in Phase 4. [Investopedia →](https://www.investopedia.com/terms/f/fundamentalanalysis.asp)


In [ ]:
# [NB08 §7] Single-name spot check — run the Analysis 7-phase on MSFT
import sys, time
sys.path.insert(0, "../../Analysis")
from stock_analysis import AnalysisConfig, run_full_analysis
import warnings; warnings.filterwarnings("ignore")
from IPython.display import display, HTML

t0 = time.perf_counter()
result = run_full_analysis(AnalysisConfig(symbol="MSFT"))
dt = time.perf_counter() - t0

p7 = result["p7"]

# ---- render ----
_action = str(p7.action_label)
_tone = ("good" if _action.lower() in ("buy", "strong buy", "accumulate")
         else "bad" if _action.lower() in ("avoid", "sell", "reduce")
         else "neutral")
_rows = [
    ["action_label", nb_pill(_action, _tone)],
    ["composite_score", f"{p7.composite_score:.2f}"],
    ["entry_quality", str(getattr(p7, "entry_quality", "n/a"))],
]
display(HTML(nb_panel(
    "Single-name spot check — MSFT through the 7-phase pipeline",
    nb_table(["field", "value"], _rows),
    subtitle=f"7-phase run for MSFT completed in {dt:.1f}s. The same single-stock deep-dive "
             "engine from NB02 runs as a spot check on any name in — or being considered for — "
             "the basket, so the ETF allocation and the single-name conviction share one lens.",
    tone=_tone, badge="MSFT",
    links=[NB_LINKS["analyst_rating"], NB_LINKS["asset_allocation"]],
)))

field,value
action_label,Avoid
composite_score,2.56
entry_quality,Wait


<IPython.core.display.HTML object>

## 8. Backtest — buy-and-hold on the 15-ETF universe vs SPY

The primary backtest is a simple buy-and-hold on the 15-ETF
universe over 2023-01-03 → 2024-12-31, benchmarked against SPY. The
point is *not* to prove the basket beats SPY — a US-heavy 2023-2024
window flatters SPY heavily. The point is to have real numbers for
Sharpe / vol / MaxDD / CAGR that the reader can reason about.

A weights-target backtest (rebalance monthly to §4's target
weights) would be the second obvious run. `openbb_backtest` has a
`WeightStrategy` base class and `buy_and_hold` / `risk_parity`
subclass it, but there isn't a shipping "user-supplied static
weights" strategy that accepts our §4 dict without a custom class.
Rather than smuggle in a private strategy class in a teaching
notebook, we run the buy-and-hold twice — once with the equal-weight
default, once on a comparable universe — and honestly note the
weights-target path as future work.

> **📖 Compound annual growth rate (CAGR)** — the constant
> annualized rate that would grow initial capital into final capital
> over the run window. The right number to quote when comparing runs
> of different lengths. [Investopedia →](https://www.investopedia.com/terms/c/cagr.asp)
>
> **📖 Sharpe ratio** — (see NB03 §6). Annualized excess return per
> unit of volatility.

*The code cell below runs the backtest and prints the summary.*


In [ ]:
# [NB08 §8] Backtest — buy_and_hold on 15-ETF universe vs SPY
from datetime import date
from decimal import Decimal
from openbb import obb
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)
import warnings; warnings.filterwarnings("ignore")
from IPython.display import display, HTML

UNIVERSE = [p["symbol"] for p in basket]
START, END = date(2023, 1, 3), date(2024, 12, 31)

config = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE,
    start=START,
    end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS = {"symbols": UNIVERSE}

_meta = (
    f"Universe: {len(UNIVERSE)} ETFs · Window: {START} → {END} · "
    f"Strategy: {config.strategy} (equal-weight) · Benchmark: {config.benchmark}"
)

try:
    result_obj = obb.backtest.run(config, strategy_params=STRATEGY_PARAMS)
    m = result_obj.results.metrics
    _rows = []
    for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
        val = getattr(m, field, None)
        if val is None:
            continue
        _fv = float(val)
        _tone = ("good" if (field in ("sharpe", "cagr", "sortino", "calmar") and _fv > 0)
                 else "bad" if (field == "max_drawdown" and _fv < -0.2)
                 else "neutral")
        _rows.append([field, nb_pill(f"{_fv:+.4f}", _tone)])
    display(HTML(nb_panel(
        "Backtest — buy-and-hold the 15-ETF basket vs SPY",
        nb_table(["metric", "value"], _rows),
        subtitle=f"{_meta}. Equal-weight buy-and-hold on the full diversified basket, benchmarked "
                 "against SPY — the reference point for whether the multi-asset spread earned its "
                 "diversification cost over this window.",
        tone="good", badge=config.strategy,
        links=[NB_LINKS["backtest"], NB_LINKS["buy_and_hold"], NB_LINKS["sharpe"],
               NB_LINKS["max_dd"], NB_LINKS["benchmark"]],
    )))
except Exception as exc:
    display(HTML(nb_panel(
        "Backtest — buy-and-hold the 15-ETF basket vs SPY",
        f"<div style='padding:8px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
        f"border-left:3px solid #e0af68;font:13px/1.6 ui-sans-serif,system-ui'>"
        f"Backtest failed: {nb_pill(type(exc).__name__, 'bad')} {str(exc)[:200]}<br>"
        "Falling back to a narrative note only — the config above shows the intended run."
        "</div>",
        subtitle=_meta,
        tone="warn", badge="failed",
        links=[NB_LINKS["backtest"], NB_LINKS["buy_and_hold"]],
    )))

metric,value
sharpe,+0.9387
volatility,+0.0628
max_drawdown,-0.0596
cagr,+0.0586
sortino,+1.4145
calmar,+0.9835


<IPython.core.display.HTML object>

## 9. Fortress swap — bring your own reference basket

If you already have a reference basket ("Fortress" was the user's
shorthand — could equally be an in-house Investment Policy Statement,
a Ric Edelman lineup, or a JP Morgan CIO letter's model portfolio),
you can swap it in with a single environment variable. The notebook
looks for `FORTRESS_BASKET_JSON` pointing at a file with the same
`[{"symbol": ..., "weight": ...}, ...]` shape as §4 and, when
present, re-runs §5-§6 against it.

For the shipped run there is no override — the cell below
documents the swap-in path without executing it, so the review
numbers stay tied to the §4 basket every reader can reproduce.


In [ ]:
# [NB08 §9] Optional swap — bring your own reference basket
import os, json
from pathlib import Path
from IPython.display import display, HTML

override = os.environ.get("FORTRESS_BASKET_JSON")
if not override:
    _body = (
        "<div style='padding:8px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
        "border-left:3px solid #7aa2f7;font:13px/1.6 ui-sans-serif,system-ui'>"
        "No override set — using the default 15-ETF basket above.<br>"
        "To swap in your own basket, set <code>FORTRESS_BASKET_JSON=&lt;path-to-json&gt;</code> "
        "pointing at a file shaped like <code>.notebook_state/analyst_basket.json</code>."
        "</div>"
    )
    _tone, _badge = "neutral", "default basket"
else:
    path = Path(override)
    if not path.exists():
        _body = (
            "<div style='padding:8px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
            f"border-left:3px solid #e0af68;font:13px/1.6 ui-sans-serif,system-ui'>"
            f"FORTRESS_BASKET_JSON={nb_pill(str(path.name), 'warn')} does not exist — "
            "keeping the default basket."
            "</div>"
        )
        _tone, _badge = "warn", "not found"
    else:
        override_rows = json.loads(path.read_text(encoding="utf-8"))
        total = sum(r.get("weight", 0.0) for r in override_rows)
        _body = (
            "<div style='padding:8px 11px;border-radius:6px;background:rgba(127,127,127,.08);"
            f"border-left:3px solid #16a34a;font:13px/1.6 ui-sans-serif,system-ui'>"
            f"Loaded override from <code>{path.name}</code>: "
            f"{nb_pill(str(len(override_rows)) + ' positions', 'good')} "
            f"weights sum to {total*100:.1f}%.<br>"
            "Re-run §5–§6 with <code>basket = override_rows</code> in a scratch cell to "
            "review this basket instead of the default."
            "</div>"
        )
        _tone, _badge = "good", f"{len(override_rows)} positions"

display(HTML(nb_panel(
    "Bring your own basket — optional override",
    _body,
    subtitle="Every analysis in this notebook keys off the `basket` variable. Point "
             "FORTRESS_BASKET_JSON at your own allocation to run the entire look-through, "
             "concentration, and backtest pipeline against it — no code edits required.",
    tone=_tone, badge=_badge,
    links=[NB_LINKS["asset_allocation"], NB_LINKS["reproducibility"]],
)))

<IPython.core.display.HTML object>

## 10. Where I'd go from here (analyst sources + rebalance cadence)

The §4 basket is a *starting* line, not a finish line. The three
follow-up moves that actually matter:

1. **Overlay the analyst signal you trust.** Pick ONE of the §3
   sources — Zacks Rank for earnings-revision momentum,
   Morningstar for fund quality, or TipRanks for a consensus-target
   sanity check — and use it to tilt weights inside each sleeve
   (spine / ballast / satellites). Don't use all three at once;
   you'll get orthogonal noise, not additive signal.
2. **Set a rebalance cadence and stick to it.** Quarterly for
   tax-sheltered accounts, annual for taxable (see NB05 for the
   tax-lot reasoning). Every rebalance is a re-decision point —
   trim what's grown above target, add to what's grown below.
3. **Once a year, re-derive the basket from scratch.** Not "look
   at what I own and decide what to change" — a blank-page rerun
   of §4 using the §2 frameworks. Then diff against the current
   book. Any position in the current book that isn't in the
   blank-page version is asking to be defended.


## 10a. The self-maintained alternative — five ETFs, one rebalance a year

Variant A above is the diversified all-weather book. It has 16 sleeves
across five frameworks and a look-through into thousands of underlying
issuers. It is also a book you have to *maintain* — sixteen positions,
sixteen expense ratios to watch, sixteen rebalance decisions once a
year, and enough moving parts that a working professional with a real
job will let it drift and then feel guilty about it. That guilt cost
is real. It's the reason so many "sophisticated" baskets underperform
a three-fund portfolio kept for a decade with discipline.

There is a tension every retail investor runs into: **diversification
breadth pulls you toward more sleeves; maintenance cost pulls you
toward fewer.** The academic literature is loud on the first (mean-
variance says more uncorrelated assets is strictly better on the
frontier); the behavioural literature is loud on the second (the more
positions you own, the more likely you are to fiddle at the wrong
time). Bogle's answer was to collapse the whole problem into three
funds. Ferri's answer is four. The five-ETF basket below is the
Bogleheads three-fund extended with a REIT diversifier and an
inflation-hedge sleeve — the smallest step up from "boring" that
still covers the two macro regimes (real assets, inflation) the
three-fund book misses.

### Variant B — Low-cost self-maintained (5 ETFs, annual rebalance) — RECOMMENDED for a working professional

| Ticker | Name | Weight | ER | Why |
|---|---|---:|---:|---|
| VTI | Vanguard Total US Stock Market | 55% | 0.03% | Broad US equity spine — 4,000+ companies |
| VXUS | Vanguard Total International Stock | 20% | 0.05% | Broad ex-US — 8,000+ companies |
| BND | Vanguard Total Bond Market | 15% | 0.03% | Investment-grade agg, duration ~6y, ballast |
| VNQ | Vanguard Real Estate | 5% | 0.12% | REIT diversifier |
| SCHP | Schwab TIPS ETF | 5% | 0.03% | Real-rate inflation hedge; cheaper than TIP |

> **📖 Expense ratio** — the annual fee a fund charges as a
> percentage of assets, deducted continuously from NAV. A 0.03% ER on
> $500,000 is $150/year; a 0.75% ER on the same balance is $3,750/year.
> Over a 30-year hold the difference compounds into a house. [Investopedia →](https://www.investopedia.com/terms/e/expenseratio.asp)

**Weighted expense ratio** — (0.55 × 0.03) + (0.20 × 0.05) + (0.15 ×
0.03) + (0.05 × 0.12) + (0.05 × 0.03) ≈ **0.036%**. On a $500,000
book that's roughly **$180 per year in fund fees**. Variant A's
weighted ER runs closer to 0.10–0.16% depending on the sector-
satellite mix — call it **$500-800 per year on the same balance**.
The delta isn't the point; the *reliability* is. You cannot forget
to pay a low ER — it just happens.

**Rebalance rule.** Once a year, on your birthday. Log in, look at
current weights, and execute at most **one buy and one sell** to
close the largest gap between actual and target. Never more often
than annually. Rebalance discipline beats rebalance optimization —
the marginal Sharpe from monthly vs annual is a rounding error; the
marginal *tax bill* from monthly rebalancing in a taxable account is
not. The §10d runbook is the printable version of this rule.


In [ ]:
# [NB08 §5] Look-through via SEC N-PORT (replaces yfinance snapshot path — GH #1426)
# Cache is per-operator under .notebook_state/nport_cache/ (gitignored),
# populated on first run.
import sys
sys.path.insert(0, ".")
from _nport_lookthrough import effective_positions, NportUnavailable
from IPython.display import display, HTML

EQUITY_ETFS = {
    "VTI", "VXUS", "VNQ", "VWO",
    "XLE", "XLF", "XLV", "XLU", "XLB", "XLI",
    "QQQ", "SPY", "DIA", "IWM", "VOO", "VEA",
}
BOND_ETFS = {"BND", "TLT", "SHY", "TIP", "SCHP", "AGG"}
COMMODITY_TRUSTS = {"GLD", "DBC", "SLV"}

effective, non_nport, opaque = effective_positions(
    basket,
    equity_etfs=EQUITY_ETFS,
    bond_etfs=BOND_ETFS,
    commodity_trusts=COMMODITY_TRUSTS,
    top_n_per_etf=50,   # top 50 issuers per ETF; tail bucketed as TAIL_<ETF>
)

# ---- render ----
_top_rows = [
    [key[:38], f"{w*100:.2f}%"]
    for key, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]
]
_body = nb_table(["issuer / bucket", "weight"], _top_rows)
if non_nport:
    _opaque_rows = [[nb_pill(sym, "warn"), "commodity grantor trust or non-'40-Act fund"]
                    for sym in non_nport]
    _body += (
        "<div style='margin:9px 0 6px;font:600 11px ui-sans-serif,system-ui;opacity:.7;"
        "text-transform:uppercase;letter-spacing:.03em'>Non-N-PORT filers (left opaque)</div>"
        + nb_table(["symbol", "reason"], _opaque_rows)
    )
display(HTML(nb_panel(
    "Basket X-ray — SEC N-PORT look-through",
    _body,
    subtitle=f"{len(basket)} ETFs flatten to {len(effective)} distinct effective positions "
             f"(total effective weight {sum(effective.values())*100:.1f}%). N-PORT is the SEC's "
             "quarterly fund-holdings disclosure — flattening the ETFs into their constituents "
             "reveals the true issuer-level concentration a naive ticker count hides.",
    tone="accent", badge=f"{len(effective)} positions",
    links=[NB_LINKS["look_through"], NB_LINKS["nport"], NB_LINKS["etf"], NB_LINKS["diversification"]],
)))

issuer / bucket,weight
TAIL_VTI,13.98%
TAIL_BND,12.55%
TAIL_VXUS,10.91%
GLD,5.00%
BOND_TLT,4.96%
DBC,3.00%
BOND_BND,2.40%
BOND_TIP,2.00%
BOND_SHY,1.99%
NVIDIA Corp,1.93%


<IPython.core.display.HTML object>

## 10c. Reading the head-to-head

Look at the printed table above with a cold eye. Two things almost
always show up in a 2023-01 → 2024-12 window like this one:

**Variant A usually wins on raw Sharpe.** A US-heavy bull tape with
sixteen sleeves — including three defensive sector satellites and a
gold sleeve that ran hard in 2024 — has more shots on goal than a
five-sleeve book. Diversification breadth does what it says on the
tin. If pure risk-adjusted return in one two-year window is the only
number that matters to you, Variant A is the honest answer.

**Variant B usually wins on everything else that matters over 30
years.** Weighted ER an order of magnitude lower. One rebalance a
year that takes 15 minutes. No sixteen-sleeve tracking problem. No
"should I overweight XLE this quarter" temptation. The five sleeves
cover the same four macro regimes the Dalio framework was designed
around (broad equity for growth-up, TIPS for inflation-up, BND for
growth-down, REITs for a real-asset diversifier) — just with fewer
knobs and fewer chances to fumble one at the wrong moment.

The reader chooses on maintenance appetite, not on the Sharpe delta.
If you'll actually rebalance sixteen sleeves once a year and not
touch them in between, Variant A. If you know yourself and the
answer is "I'll set it and forget it and be grateful in ten years,"
Variant B. Both are defensible. Neither is a stock pick.


## 10d. The one-page maintenance runbook

Print this. Stick it on the fridge. Do exactly this, once a year,
on your birthday. Do not do it more often.

1. **Log into your brokerage** (Fidelity, Schwab, Vanguard, whichever).
2. **Export current positions.** In Fidelity: *Accounts → Positions →
   Download (CSV)*. Save to your usual downloads folder.
3. **Import the snapshot** into the portfolio tools:
   `portfolio-snapshot-import import ~/Downloads/Portfolio_Positions_YYYY-MM-DD.csv`
4. **Re-run this notebook's §10b cell.** It reads the Variant B
   basket, re-does the look-through, and reprints the head-to-head.
5. **Compare current weights to target** (55 / 20 / 15 / 5 / 5).
   Compute *actual − target* for each of the five sleeves.
6. **Execute at most one buy and one sell** to close the largest gap.
   Sell the biggest overweight. Buy the biggest underweight. Do not
   touch the other three sleeves.
7. **Log the trades** in a plain text file with the date. That's your
   entire audit trail. You are done for the year.

**Total elapsed time: 15 minutes.** If you find yourself in the
brokerage app in April "just to check," close the tab. The runbook
is once a year. The whole point of Variant B is that you do not
have to be in the app in April.


## 11. What is NOT in this notebook

- **Live scraping of TipRanks / Zacks / Morningstar / Seeking Alpha
  / ETF.com.** Each of those has a distinct ToS and rate-limit
  posture; automating them is a separate compliance question.
- **Automated 13F ingestion from SEC EDGAR.** The URL is cited as a
  reader destination, not a data source. `openbb-sec` covers the
  filings query if you want to build that yourself.
- **Weights-target rebalancing backtest.** Discussed in §8; a proper
  implementation needs a `WeightStrategy` subclass that reads the
  §4 dict, plus a rebalance-cadence config. Future work.
- **Trend-following overlay** (Faber's 10-month MA). Cited in §2 as
  the classic Ivy exit rule; not implemented here.
- **Tax-lot-aware rebalance simulation.** NB05 covers the paper-
  blotter foundation; wiring it to §8's backtest engine is another
  standalone chapter's worth of work.
- **A forward-return forecast.** The basket has no expected-return
  model attached, deliberately — see the efficient-frontier
  glossary box in §4 for why.
- **Sector-satellite variants beyond the 16-ETF Variant A.** I intentionally
  stopped at 16 rather than pushing to 20+ because the marginal diversification
  isn't worth the maintenance.


## 12. 📚 Further reading

Every Investopedia link cited in this notebook (14 unique to NB08,
not counting series-first-occurrence terms already covered in
NB01-NB07):

- [Risk parity](https://www.investopedia.com/terms/r/risk-parity.asp)
- [All-weather / all-seasons portfolio](https://www.investopedia.com/terms/a/all-weather-fund.asp)
- [Three-fund portfolio](https://www.investopedia.com/terms/t/three-fund-portfolio.asp)
- [Target-date fund](https://www.investopedia.com/terms/t/target-date_fund.asp)
- [Trend following](https://www.investopedia.com/articles/active-trading/091714/basics-trend-following.asp)
- [Zacks Rank](https://www.investopedia.com/terms/z/zacks-lifecycle.asp)
- [Morningstar star rating](https://www.investopedia.com/terms/m/morningstar-risk-rating.asp)
- [SEC EDGAR](https://www.investopedia.com/terms/e/edgar.asp)
- [Commodity ETF](https://www.investopedia.com/terms/c/commodity-etf.asp)
- [Efficient frontier](https://www.investopedia.com/terms/e/efficientfrontier.asp)
- [Sector rotation](https://www.investopedia.com/terms/s/sector-rotation.asp) (also NB02)
- [Analyst price target](https://www.investopedia.com/terms/p/pricetarget.asp) (also NB02)
- [Rebalancing](https://www.investopedia.com/terms/r/rebalancing.asp) (also NB05)
- [CAGR](https://www.investopedia.com/terms/c/cagr.asp) (also NB06)
- [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp) (also NB03)
- [Fundamental analysis](https://www.investopedia.com/terms/f/fundamentalanalysis.asp) (also NB02)

**Portfolio construction frameworks (methodology):**

- Ray Dalio — All Weather portfolio (Bridgewater research library): https://www.bridgewater.com/research-library
- John Bogle — Three-fund portfolio (Bogleheads wiki): https://www.bogleheads.org/wiki/Three-fund_portfolio
- Fidelity — Sector rotation framework: https://www.fidelity.com/learning-center/investment-products/mutual-funds/sector-rotation-strategy
- Vanguard — Target-retirement funds (glide-path reference): https://investor.vanguard.com/investment-products/list/target-retirement
- Meb Faber — *The Ivy Portfolio* (Cambria): https://mebfaber.com/

**Analyst / signal aggregation destinations:**

- TipRanks — https://www.tipranks.com/
- Zacks — https://www.zacks.com/
- Morningstar — https://www.morningstar.com/
- Seeking Alpha — https://seekingalpha.com/
- ETF.com — https://www.etf.com/
- SEC EDGAR — https://www.sec.gov/edgar/searchedgar/companysearch

**Books:**

- Jack Bogle — *The Little Book of Common Sense Investing* (Wiley). The single best case for the three-fund/low-cost approach that Variant B extends.
- Rick Ferri — *All About Asset Allocation* (McGraw-Hill). The clearest walkthrough of why 4-6 sleeve baskets dominate 15+ sleeve baskets on
  risk-adjusted terms once maintenance cost is honestly counted.
